# Shortest Path

Simulação do uso de grafos em contexto de controle

Crie uma conta gratuita no Neo4J Aura em: https://neo4j.com/cloud/platform/auradb

Em seguida, crie um instancia de banco e obtenha os dados abaixo:   

Este notebook demonstra a aplicação de grafos para identificar caminhos mais curtos e potenciais conflitos de interesse em contextos de auditoria. Utilizaremos o Neo4j, um banco de dados de grafos, para modelar relações entre pessoas, órgãos, empresas e licitações.

### Pré-requisitos:
1.  **Conta Neo4j Aura**: Crie uma conta gratuita em [https://neo4j.com/cloud/platform/auradb](https://neo4j.com/cloud/platform/auradb).
2.  **Instância Neo4j**: Após criar a conta, crie uma instância de banco de dados e obtenha as credenciais (URI, Usuário, Senha) para preencher as variáveis abaixo.
3.  **Dados CSV**: Certifique-se de ter os arquivos CSV com os dados fictícios disponíveis no seu Google Drive, conforme o caminho configurado posteriormente.

In [1]:
import os
from dotenv import load_dotenv

load_dotenv(dotenv_path='.env')

NEO4J_URI=os.getenv("NEO4J_URI")
NEO4J_USERNAME=os.getenv("NEO4J_USERNAME")
NEO4J_DATABASE=os.getenv("NEO4J_DATABASE")
AURA_INSTANCEID=os.getenv("AURA_INSTANCEID")
AURA_INSTANCENAME=os.getenv("AURA_INSTANCENAME")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")


### 1. Configuração das Credenciais do Neo4j

Nesta célula, você deve inserir as credenciais da sua instância do Neo4j Aura DB. Substitua os valores pelos que você obteve na plataforma Neo4j. Estes dados são essenciais para que o notebook consiga se conectar ao seu banco de dados de grafos.

Instale o pacote Neo4J para Python:

### 2. Instalação do Driver Python para Neo4j

Para interagir com o banco de dados Neo4j a partir do Python, precisamos instalar a biblioteca `neo4j`. O comando `!pip install neo4j` faz essa instalação no ambiente do Colab.

In [2]:
!pip install neo4j

Defaulting to user installation because normal site-packages is not writeable


### 3. Importação de Bibliotecas e Montagem do Google Drive

Esta célula realiza a importação das bibliotecas necessárias (`pandas` para manipulação de dados, `GraphDatabase` do `neo4j` para conexão e `google.colab.drive` para acesso ao Drive). Além disso, monta o seu Google Drive, o que permite que o notebook acesse os arquivos CSV que contêm os dados a serem carregados no grafo. **Lembre-se de ajustar o `CSV_DIR` para o caminho correto onde seus arquivos CSV estão salvos no Drive.**

In [3]:
import pandas as pd
from neo4j import GraphDatabase
#from google.colab import drive
import os

# Montar o Google Drive
#drive.mount('/content/drive')

# ⚠️ AJUSTE AQUI O CAMINHO DA SUA PASTA NO GOOGLE DRIVE
#CSV_DIR = '/content/drive/MyDrive/Aulas/Aulas IDP/2026/Auditoria de Dados e Accountability/modulo IV - Grafos e IA/grafos'

CSV_DIR="./grafos"

print("Conteúdo da pasta de CSVs:")
if os.path.exists(CSV_DIR):
    print(os.listdir(CSV_DIR))
else:
    print(f"⚠️ Atenção: A pasta '{CSV_DIR}' ainda não existe no seu Google Drive. Verifique o caminho!")

Conteúdo da pasta de CSVs:
['relacao_socios.csv', 'relacao_pai_filho.csv', 'orgaos_ficticios.csv', 'licitacoes.csv', 'pessoas_ficticias.csv', 'relacao_mae_filho.csv', 'participantes_licitacao.csv', 'empresas_ficticias.csv', 'relacao_pregoeiros.csv']


Testando a instância

### 4. Teste de Conexão e Limpeza do Banco de Dados

Este bloco de código estabelece uma conexão inicial com o banco de dados Neo4j usando as credenciais fornecidas. A linha `driver.verify_connectivity()` confirma se a conexão foi bem-sucedida. Em seguida, ele executa uma consulta Cypher (`MATCH (n) DETACH DELETE n`) para **apagar todos os nós e relacionamentos existentes** no banco de dados. Isso é útil para garantir um ambiente limpo antes de carregar novos dados, especialmente em desenvolvimento e testes. **Cuidado ao usar em produção!**

Verifique no Neo4J Aura Bloom o estado do seu grafo

In [4]:
with GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME,NEO4J_PASSWORD)) as driver:

    driver.verify_connectivity()

    # Clear existing data before adding new nodes/relationships
    with driver.session() as session:
        session.run("MATCH (n) DETACH DELETE n")
        print("Existing nodes and relationships deleted.")



Existing nodes and relationships deleted.


### 5. Criação da Classe `Neo4jConn` para Interação com o Banco de Dados

Para facilitar a interação com o Neo4j, criamos uma classe `Neo4jConn`. Ela encapsula a lógica de conexão (`__init__`) e o método `query` para executar comandos Cypher. Isso torna o código mais organizado e reutilizável. O teste de conexão final verifica se a classe consegue se comunicar e obter a versão do Neo4j.

In [5]:
class Neo4jConn:
    def __init__(self, uri, user, password):
        self.driver = GraphDatabase.driver(uri, auth=(user, password))

    def close(self):
        self.driver.close()

    def query(self, query_str, parameters=None):
        with self.driver.session() as session:
            result = session.run(query_str, parameters or {})
            return [record.data() for record in result]

# Teste de Conexão
db = Neo4jConn(NEO4J_URI, NEO4J_USERNAME, NEO4J_PASSWORD)
print("Conexão bem-sucedida! Versão do Neo4j:", db.query("CALL dbms.components() YIELD name, versions, edition WHERE name = 'Neo4j Kernel' RETURN versions[0] AS neo4jVersion"))

Conexão bem-sucedida! Versão do Neo4j: [{'neo4jVersion': '5.27-aura'}]


### 6. Criação de Restrições de Unicidade

As restrições de unicidade são cruciais para garantir a integridade dos dados no grafo. Elas asseguram que não haverá nós duplicados para propriedades que devem ser únicas (como `cpf` para `Pessoa`, `cnpj` para `Empresa`, etc.). Isso otimiza o desempenho das consultas e a coerência do modelo de dados.

In [6]:
# ==========================================
# 3. CRIAÇÃO DE RESTRIÇÕES DE UNICIDADE
# ==========================================
# Opcional: Limpar banco antes da carga (CUIDADO em ambientes de produção)
# db.query("MATCH (n) DETACH DELETE n")

constraints = [
    "CREATE CONSTRAINT IF NOT EXISTS FOR (o:Orgao) REQUIRE o.codigo_orgao IS UNIQUE;",
    "CREATE CONSTRAINT IF NOT EXISTS FOR (p:Pessoa) REQUIRE p.cpf IS UNIQUE;",
    "CREATE CONSTRAINT IF NOT EXISTS FOR (e:Empresa) REQUIRE e.cnpj IS UNIQUE;",
    "CREATE CONSTRAINT IF NOT EXISTS FOR (l:Licitacao) REQUIRE l.codigo_licitacao IS UNIQUE;"
]

for c in constraints:
    db.query(c)

print("Restrições de unicidade criadas com sucesso!")

Restrições de unicidade criadas com sucesso!


### 7. Carga dos Nós (Entidades)

Nesta etapa, carregamos os dados dos arquivos CSV para criar os diferentes tipos de nós no grafo: Órgãos, Pessoas, Empresas e Licitações. O `UNWIND $rows AS row` permite processar múltiplos registros de uma vez, e o `MERGE` cria o nó se ele não existir, ou encontra-o se já existir, e `SET` define suas propriedades. No caso das licitações, também criamos o relacionamento `PROMOVIDA_POR` com o órgão correspondente.

In [7]:
# ==========================================
# 4. CARGA DOS NÓS
# ==========================================

# 4.1 Carregar Órgãos
df_orgaos = pd.read_csv(os.path.join(CSV_DIR, "orgaos_ficticios.csv"))
query_orgaos = """
UNWIND $rows AS row
MERGE (o:Orgao {codigo_orgao: row.codigo_orgao})
SET o.nome_orgao = row.nome_orgao
"""
db.query(query_orgaos, {"rows": df_orgaos.to_dict('records')})
print(f"✓ Órgãos carregados: {len(df_orgaos)}")


# 4.2 Carregar Pessoas
df_pessoas = pd.read_csv(os.path.join(CSV_DIR, "pessoas_ficticias.csv"))
query_pessoas = """
UNWIND $rows AS row
MERGE (p:Pessoa {cpf: row.cpf})
SET p.nome = row.nome, p.data_nascimento = row.data_nascimento
"""
db.query(query_pessoas, {"rows": df_pessoas.to_dict('records')})
print(f"✓ Pessoas carregadas: {len(df_pessoas)}")

# 4.3 Carregar Empresas
df_empresas = pd.read_csv(os.path.join(CSV_DIR, "empresas_ficticias.csv"))
query_empresas = """
UNWIND $rows AS row
MERGE (e:Empresa {cnpj: row.cnpj})
SET e.razao_social = row.razao_social
"""
db.query(query_empresas, {"rows": df_empresas.to_dict('records')})
print(f"✓ Empresas carregadas: {len(df_empresas)}")

# 4.4 Carregar Licitações e vincular ao Órgão (PROMOVIDA_POR)
df_licitacoes = pd.read_csv(os.path.join(CSV_DIR, "licitacoes.csv"))
query_licitacoes = """
UNWIND $rows AS row
MERGE (l:Licitacao {codigo_licitacao: row.codigo_licitacao})
SET l.objeto = row.objeto,
    l.unidade_medida = row.unidade_medida,
    l.quantidade = row.quantidade,
    l.valor_unitario_estimado = row.valor_unitario_estimado,
    l.valor_total_estimado = row.valor_total_estimado
WITH l, row
MATCH (o:Orgao {codigo_orgao: row.codigo_orgao})
MERGE (l)-[:PROMOVIDA_POR]->(o)
"""
db.query(query_licitacoes, {"rows": df_licitacoes.to_dict('records')})
print(f"✓ Licitações carregadas e vinculadas aos Órgãos: {len(df_licitacoes)}")

✓ Órgãos carregados: 20
✓ Pessoas carregadas: 100
✓ Empresas carregadas: 50
✓ Licitações carregadas e vinculadas aos Órgãos: 60


### 8. Carga dos Relacionamentos (Arestas)

Após carregar todos os nós, esta seção é dedicada à criação dos relacionamentos que conectam essas entidades. Os relacionamentos são a essência de um banco de dados de grafos e representam as interações e vínculos entre os nós.

-   **É PREGOEIRO DE**: Conecta uma `Pessoa` a um `Orgao` onde ela atua como pregoeiro.
-   **É SÓCIO DE**: Liga uma `Pessoa` a uma `Empresa` da qual ela é sócia.
-   **É MÃE DE / É PAI DE**: Estabelece vínculos de parentesco entre `Pessoa`s.
-   **PARTICIPOU DE**: Conecta uma `Empresa` a uma `Licitacao` na qual ela participou, indicando se foi a vencedora e o valor da proposta.

In [8]:
# ==========================================
# 5. CARGA DOS RELACIONAMENTOS (ARESTAS)
# ==========================================

# 5.1 Relação: É PREGOEIRO DE
df_pregoeiros = pd.read_csv(os.path.join(CSV_DIR, "relacao_pregoeiros.csv"))
q_pregoeiros = """
UNWIND $rows AS row
MATCH (p:Pessoa {cpf: row.cpf_pregoeiro})
MATCH (o:Orgao {codigo_orgao: row.codigo_orgao})
MERGE (p)-[r:E_PREGOEIRO_DE]->(o)
SET r.data_designacao = row.data_designacao,
    r.portaria_designacao = row.portaria_designacao
"""
db.query(q_pregoeiros, {"rows": df_pregoeiros.to_dict('records')})
print(f"✓ Vínculos de Pregoeiros criados: {len(df_pregoeiros)}")

# 5.2 Relação: É SÓCIO DE
df_socios = pd.read_csv(os.path.join(CSV_DIR, "relacao_socios.csv"))
q_socios = """
UNWIND $rows AS row
MATCH (p:Pessoa {cpf: row.cpf_socio})
MATCH (e:Empresa {cnpj: row.cnpj_empresa})
MERGE (p)-[r:E_SOCIO_DE]->(e)
SET r.qualificacao_socio = row.qualificacao_socio,
    r.percentual_participacao = row.percentual_participacao,
    r.data_entrada_sociedade = row.data_entrada_sociedade
"""
db.query(q_socios, {"rows": df_socios.to_dict('records')})
print(f"✓ Vínculos Societários criados: {len(df_socios)}")

# 5.3 Relações Parentais (É MÃE DE / É PAI DE)
df_mae = pd.read_csv(os.path.join(CSV_DIR, "relacao_mae_filho.csv"))
q_mae = """
UNWIND $rows AS row
MATCH (m:Pessoa {cpf: row.cpf_mae})
MATCH (f:Pessoa {cpf: row.cpf_filho})
MERGE (m)-[:E_MAE_DE]->(f)
"""
db.query(q_mae, {"rows": df_mae.to_dict('records')})

df_pai = pd.read_csv(os.path.join(CSV_DIR, "relacao_pai_filho.csv"))
q_pai = """
UNWIND $rows AS row
MATCH (p:Pessoa {cpf: row.cpf_pai})
MATCH (f:Pessoa {cpf: row.cpf_filho})
MERGE (p)-[:E_PAI_DE]->(f)
"""
db.query(q_pai, {"rows": df_pai.to_dict('records')})
print(f"✓ Vínculos de Parentesco (Mãe/Pai) criados: {len(df_mae) + len(df_pai)}")

# 5.4 Relação: PARTICIPOU DE (Empresa -> Licitação)
df_part = pd.read_csv(os.path.join(CSV_DIR, "participantes_licitacao.csv"))
q_part = """
UNWIND $rows AS row
MATCH (e:Empresa {cnpj: row.cnpj_empresa})
MATCH (l:Licitacao {codigo_licitacao: row.codigo_licitacao})
MERGE (e)-[r:PARTICIPOU_DE]->(l)
SET r.e_vencedora = row.e_vencedora,
    r.valor_proposta = row.valor_proposta
"""
db.query(q_part, {"rows": df_part.to_dict('records')})
print(f"✓ Propostas e Participações em Licitações criadas: {len(df_part)}")

✓ Vínculos de Pregoeiros criados: 29
✓ Vínculos Societários criados: 144
✓ Vínculos de Parentesco (Mãe/Pai) criados: 31
✓ Propostas e Participações em Licitações criadas: 209


### 9. Consultas de Auditoria: `shortestPath` (Caminho Mais Curto)

Esta é a parte central da simulação de auditoria. A consulta Cypher `shortestPath` é utilizada para encontrar o caminho mais curto entre um `Pregoeiro` e o `Sócio` de uma `Empresa` que venceu uma licitação no mesmo `Órgão` do pregoeiro. O objetivo é identificar possíveis relações de proximidade (diretas ou indiretas) que poderiam indicar conflitos de interesse ou conluio.

-   `MATCH (pregoeiro:Pessoa)-[:E_PREGOEIRO_DE]->(orgao:Orgao)<-[:PROMOVIDA_POR]-(lic:Licitacao)<-[p:PARTICIPOU_DE {e_vencedora: true}]-(empresa:Empresa)<-[:E_SOCIO_DE]-(socio:Pessoa)`: Este padrão inicial identifica a relação de interesse: um pregoeiro de um órgão, uma licitação promovida por esse órgão e vencida por uma empresa que tem um sócio.
-   `WHERE pregoeiro <> socio`: Garante que o pregoeiro e o sócio são pessoas distintas.
-   `MATCH path = shortestPath((pregoeiro)-[*..6]-(socio))`: Procura o caminho mais curto entre o pregoeiro e o sócio, limitado a 6 'saltos' (relacionamentos).
-   `RETURN`: Retorna informações detalhadas sobre o achado, incluindo o caminho percorrido e a distância em 'saltos'.

In [9]:
# ==========================================
# 6. CONSULTAS DE AUDITORIA: SHORTEST PATH
# ==========================================

# CONSULTA 1: Menor caminho entre qualquer Pregoeiro e os Sócios de Empresas que VENCERAM licitações no seu órgão
cypher_shortest_path = """
MATCH (pregoeiro:Pessoa)-[:E_PREGOEIRO_DE]->(orgao:Orgao)<-[:PROMOVIDA_POR]-(lic:Licitacao)<-[p:PARTICIPOU_DE {e_vencedora: true}]-(empresa:Empresa)<-[:E_SOCIO_DE]-(socio:Pessoa)
WHERE pregoeiro <> socio

// Busca o Menor Caminho não direcionado entre o Pregoeiro e o Sócio da empresa vencedora (ignorando a rota direta pela própria licitação)
MATCH path = shortestPath((pregoeiro)-[*..6]-(socio))

RETURN
    orgao.codigo_orgao AS Orgao,
    lic.codigo_licitacao AS Licitacao,
    empresa.razao_social AS EmpresaVencedora,
    pregoeiro.nome AS Pregoeiro,
    socio.nome AS SocioEmpresa,
    length(path) AS DistanciaSaltos,
    [node IN nodes(path) | coalesce(node.nome, node.razao_social, node.nome_orgao, node.codigo_licitacao)] AS Caminho
ORDER BY DistanciaSaltos ASC
LIMIT 10
"""

resultados = db.query(cypher_shortest_path)

print(f"🔍 [AUDITORIA] Foram encontrados {len(resultados)} caminhos suspeitos de proximidade!\n")

for i, res in enumerate(resultados, 1):
    print(f"--- Achado de Auditoria #{i} ---")
    print(f"🏛️ Órgão: {res['Orgao']}")
    print(f"📜 Licitação: {res['Licitacao']} | Empresa Vencedora: {res['EmpresaVencedora']}")
    print(f"👤 Pregoeiro: {res['Pregoeiro']} <---> 👔 Sócio: {res['SocioEmpresa']}")
    print(f"📏 Distância do Caminho: {res['DistanciaSaltos']} saltos")
    print(f"🔗 Sequência do Caminho: {' ➔ '.join(res['Caminho'])}\n")

🔍 [AUDITORIA] Foram encontrados 10 caminhos suspeitos de proximidade!

--- Achado de Auditoria #1 ---
🏛️ Órgão: ORG-019
📜 Licitação: LIC-2024-0059 | Empresa Vencedora: Alpha Suprimentos e Serviços Eireli
👤 Pregoeiro: Mateus Oliveira Santos <---> 👔 Sócio: Vanessa Araújo Cavalcanti
📏 Distância do Caminho: 2 saltos
🔗 Sequência do Caminho: Mateus Oliveira Santos ➔ Rafaela Costa Ribeiro ➔ Vanessa Araújo Cavalcanti

--- Achado de Auditoria #2 ---
🏛️ Órgão: ORG-001
📜 Licitação: LIC-2024-0004 | Empresa Vencedora: Horizonte Comunicação e Serviços S.A.
👤 Pregoeiro: Mateus Santana Ramos <---> 👔 Sócio: Lucas Mendes Teixeira
📏 Distância do Caminho: 3 saltos
🔗 Sequência do Caminho: Mateus Santana Ramos ➔ Sergio Oliveira Alves ➔ Apex Tecnologia e Serviços Eireli ➔ Lucas Mendes Teixeira

--- Achado de Auditoria #3 ---
🏛️ Órgão: ORG-002
📜 Licitação: LIC-2024-0023 | Empresa Vencedora: Delta Consultoria ME
👤 Pregoeiro: Adriana Mendes Cavalcanti <---> 👔 Sócio: Ingrid Dias Alves
📏 Distância do Caminho: 3 s

### 10. Visualização da Query no Neo4j Bloom ou Browser

A consulta Cypher fornecida nesta célula é ideal para ser executada diretamente na interface gráfica do Neo4j, como o Neo4j Browser ou o Neo4j Bloom (disponível no Aura DB). Ao colar e executar essa query lá, você poderá visualizar o grafo resultante dos caminhos suspeitos, com os nós e relacionamentos destacados. Isso facilita a compreensão das conexões e a identificação visual de padrões de fraude ou conluio. A cláusula `RETURN path, pregoeiro, socio, lic, empresa, orgao, r_pregoeiro_orgao, r_socio_empresa, r_promovida, p, length(path) AS DistanciaSaltos` garante que todos os elementos do caminho e suas propriedades sejam exibidos na visualização.

Cole essa query no Neo4J:
```cypher
MATCH (pregoeiro:Pessoa)-[r_pregoeiro_orgao:E_PREGOEIRO_DE]->(orgao:Orgao)<-[r_promovida:PROMOVIDA_POR]-(lic:Licitacao)<-[p:PARTICIPOU_DE {e_vencedora: true}]-(empresa:Empresa)<-[r_socio_empresa:E_SOCIO_DE]-(socio:Pessoa)
WHERE pregoeiro <> socio
MATCH path = shortestPath((pregoeiro)-[*..6]-(socio))
RETURN path, pregoeiro, socio, lic, empresa, orgao,  r_pregoeiro_orgao, r_socio_empresa, r_promovida, p,
length(path) AS DistanciaSaltos
ORDER BY DistanciaSaltos ASC
LIMIT 3
```